# Student Performance Model Training 
This notebook provides a concise, reproducible view of the model-training workflow. The authoritative application code remains in `src/`: this notebook is supporting evidence for dataset inspection, model comparison, evaluation, and a sample advisory.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.sri_lankan_schema import (
    LOCAL_CATEGORICAL_FEATURES,
    LOCAL_FEATURES,
    LOCAL_NUMERIC_FEATURES,
    categorize_gpa,
)

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "sri_lankan_university.csv"
data = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {data.shape}")
display(data.head())

Dataset shape: (360, 17)


,academic_year,semester,faculty,programme,previous_gpa,attendance_percentage,study_hours_per_week,failed_modules,assignment_completion_percentage,assessment_average_percentage,lecture_participation_percentage,tutorial_participation,lms_active,internet_access,financial_work_pressure,wellbeing_rating,semester_gpa
0,4,2,Computing,Information Technology,3.19,87.6,6,0,58.4,86.3,78.8,yes,no,yes,no,3,2.31
1,1,2,Management,Accounting,3.01,73.0,13,0,82.3,80.4,82.6,yes,yes,yes,yes,4,2.44
2,3,2,Humanities,Social Sciences,2.11,76.0,9,0,92.0,75.0,60.9,yes,no,yes,no,4,1.96
3,4,1,Management,Accounting,3.12,84.4,12,0,91.1,89.6,82.9,no,yes,yes,yes,2,3.07
4,3,1,Management,Business Management,2.28,66.8,16,0,96.0,92.2,71.3,yes,yes,limited,no,2,2.46


## Dataset and target inspection

The model uses academic, participation, access, and wellbeing indicators. The target is derived from `semester_gpa` using the same categorization function used by the training script.

In [ ]:
print("Missing values:")
display(data.isna().sum().to_frame("missing"))

data["performance_category"] = data["semester_gpa"].apply(categorize_gpa)
print("Performance categories:")
display(data["performance_category"].value_counts().rename_axis("category").to_frame("records"))

print("Model input features:")
print(LOCAL_FEATURES)

## Train and compare models

The workflow uses an 80/20 stratified split with `random_state=42`. Numeric features are standardized and categorical features are one-hot encoded inside each pipeline. The highest-accuracy model on the held-out test data is selected.

In [ ]:
train_features, test_features, train_target, test_target = train_test_split(
    data[LOCAL_FEATURES],
    data["performance_category"],
    test_size=0.2,
    random_state=42,
    stratify=data["performance_category"],
)

estimators = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=8, min_samples_split=5, class_weight="balanced", random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=250,
        max_depth=10,
        min_samples_split=5,
        random_state=42,
        class_weight="balanced",
    ),
}

pipelines = {}
scores = {}
for name, estimator in estimators.items():
    preprocessor = ColumnTransformer([
        ("num", StandardScaler(), LOCAL_NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), LOCAL_CATEGORICAL_FEATURES),
    ])
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimator),
    ])
    pipeline.fit(train_features, train_target)
    pipelines[name] = pipeline
    scores[name] = accuracy_score(test_target, pipeline.predict(test_features))

comparison = pd.DataFrame({"model": list(scores), "accuracy": list(scores.values())})
display(comparison.sort_values("accuracy", ascending=False).reset_index(drop=True))
selected_name = max(scores, key=scores.get)
selected_model = pipelines[selected_name]
print(f"Selected model: {selected_name}")

## Evaluation 

In [ ]:
predictions = selected_model.predict(test_features)
print(f"Held-out records: {len(test_target)}")
print(f"Accuracy: {accuracy_score(test_target, predictions):.3f}")
print("\nClassification report:")
print(classification_report(test_target, predictions, zero_division=0))
print("Confusion matrix (rows = actual, columns = predicted):")
labels = ["At Risk", "Average", "High Performance"]
display(pd.DataFrame(
    confusion_matrix(test_target, predictions, labels=labels),
    index=labels,
    columns=labels,
))

## Sample prediction and rule-based advisory

The machine-learning model predicts the performance category. The separate advisory module then applies transparent rules to generate actions such as attendance improvement or failed-module recovery.

In [ ]:
from src.university_advisory import generate_advisory

sample_student = {
    "academic_year": 2,
    "semester": 1,
    "faculty": "Computing",
    "programme": "Computer Science",
    "previous_gpa": 1.7,
    "attendance_percentage": 62,
    "study_hours_per_week": 5,
    "failed_modules": 2,
    "assignment_completion_percentage": 48,
    "assessment_average_percentage": 35,
    "lecture_participation_percentage": 55,
    "tutorial_participation": "no",
    "lms_active": "no",
    "internet_access": "limited",
    "financial_work_pressure": "yes",
    "wellbeing_rating": 2,
}

sample_prediction = selected_model.predict(pd.DataFrame([sample_student]))[0]
sample_probabilities = dict(zip(selected_model.classes_, selected_model.predict_proba(pd.DataFrame([sample_student]))[0]))
sample_advisory = generate_advisory(sample_student, sample_prediction)

print(f"Prediction: {sample_prediction}")
print("Probabilities:", {label: round(value, 3) for label, value in sample_probabilities.items()})
display(pd.DataFrame(sample_advisory["impact_analysis"]["priorities"]))